# 🚀 FaceSwapperVideoV1 - Colab Optimized Notebook
This notebook runs **FaceSwapperVideoV1** on Google Colab (Free T4 GPU) with optimized speed and high-quality face restoration.

### Step 1: Check GPU & Clone Repository

In [ ]:
!nvidia-smi
!git clone https://github.com/Deci1337/FaceSwapperVideoV1.git
%cd FaceSwapperVideoV1

### Step 2: Install Dependencies (CUDA Accelerated)

In [ ]:
!pip install -r requirements.txt
!pip install onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!pip install gradio codeformer-fidelity

### Step 3: Launch Gradio Web UI (Accessible via Android Browser/Termux)

In [ ]:
import gradio as gr
import subprocess
import os

def run_faceswap(source_img, target_vid, enhancer, fidelity):
    if not source_img or not target_vid:
        return None
    
    output_path = "/content/output_result.mp4"
    
    cmd = [
        "python", "cli.py",
        "--source", source_img,
        "--target", target_vid,
        "--output", output_path,
        "--execution-provider", "cuda",
        "--execution-threads", "8",
        "--color-matching", "seamless"
    ]
    
    if enhancer != "none":
        cmd.extend(["--face-enhancer", enhancer, "--codeformer-fidelity", str(fidelity)])
        
    subprocess.run(cmd)
    return output_path

with gr.Blocks(title="FaceSwapper Cloud UI") as demo:
    gr.Markdown("## 🚀 FaceSwapperVideoV1 (Colab T4 Optimized)")
    with gr.Row():
        with gr.Column():
            source_input = gr.Image(type="filepath", label="Source Face Image")
            target_input = gr.Video(label="Target Video")
            enhancer_opt = gr.Radio(["none", "codeformer", "gfpgan"], value="codeformer", label="Face Enhancer Quality")
            fidelity_slider = gr.Slider(0.1, 1.0, value=0.8, label="CodeFormer Fidelity (0.8 Recommended)")
            btn = gr.Button("Start Face Swap", variant="primary")
        with gr.Column():
            video_output = gr.Video(label="Output Result")
            
    btn.click(run_faceswap, inputs=[source_input, target_input, enhancer_opt, fidelity_slider], outputs=video_output)

demo.launch(share=True, debug=True)